# Solar Filament Segmentation, notebook 03: U-Net, UNet++ and DeepLabV3+ trained on full-resolution crops

COSC 50523 Deep Learning (TCU), course project. Dataset: MAGFiLO (Kaggle competition `filament-segmentation-2026`).

This is part 1 of the plan in our midterm report. Notebook 02 shrank every image from 2048 x 2048 to 512 x 512 before training, and at that size many thin filament threads are one or two pixels wide or gone. In this notebook the U-Net trains on 512 x 512 crops cut from the full 2048 x 2048 images and predicts a full image tile by tile. Each image is decoded once into a disk cache (`CACHE_DIR`), and a crop is read from that cache only when a batch needs it. So the roughly 19 GB that full-size float arrays would take never has to fit in RAM.

The notebook trains four models. The first two are U-Nets that differ only in resolution (part 1 of the plan):

| Run | What the U-Net sees in training | How a full image is predicted |
|---|---|---|
| `resized_512` | the whole image resized to 512 x 512 (the midterm setting) | one pass at 512, probabilities upsampled to 2048 |
| `crops_2048` | 512 x 512 crops from the 2048 x 2048 image, half of them placed on a filament | 25 overlapping 512 x 512 tiles, blended |

The rest is shared, and most of it is copied from notebook 02: the day-based split (same code, same seed), a ResNet34 encoder pretrained on ImageNet, weighted BCE plus soft Dice loss, Adam at 3e-4, batch 8, 15 epochs, the same augmentation, best epoch by validation Dice, a threshold tuned on validation, and one scoring of the test set at the end. Both runs draw `CROPS_PER_IMAGE` = 4 samples per training image per epoch, so they take the same number of gradient steps. For `resized_512` that means each image is seen four times per epoch with different augmentation, which is four times the training notebook 02 did. Its result is therefore a stronger reference point than the midterm number.

**Part 2: a second architecture.** The other two runs keep everything from `crops_2048` (crops, sampling, encoder, loss, optimizer, epochs) and change only the decoder, so any difference to the crop U-Net comes from the architecture:

| Run | Architecture | Idea |
|---|---|---|
| `unetpp_crops_2048` | UNet++ (Zhou et al., 2018), ResNet34 encoder | extra convolution blocks on every skip connection, prediction at full resolution: aimed at fine detail and thin structures |
| `deeplabv3p_crops_2048` | DeepLabV3+ (Chen et al., 2018), ResNet34 encoder | dilated convolutions and an ASPP module for much wider context, but the prediction is made at 1/4 resolution and upsampled 4x, so thin threads may blur |

The two pull in opposite directions (detail versus context), and the recall-by-width table in section 11 shows where each one gains or loses.

**Metrics.** Pooled IoU, Dice, precision and recall over all pixels of the held-out images, as in notebooks 01 and 02. We compute them twice. The main numbers use masks drawn at 2048 x 2048, the size of the original images. The second set uses masks drawn at 512 x 512 with the same code as notebook 02, so they can go next to the midterm table (U-Net test IoU 0.5331, Dice 0.6955). Each mask size gets its own threshold, tuned on validation. On top of that we measure recall for every annotated filament in the test set and group the filaments by estimated width, which checks directly whether thin filaments are found more often at full resolution.

**Configuration.** The values below come from the config cell. `EPOCHS`, `BATCH`, `CROPS_PER_IMAGE`, `NUM_WORKERS`, `ENCODER_WEIGHTS`, `RUNS`, `DATA_DIR`, `CACHE_DIR`, `OUTPUT_DIR`, `SAVE_TO_DRIVE`, `RESUME` and `REUSE_WEIGHTS` can also be set as environment variables.

| Setting | Value |
|---|---|
| Model | `smp.create_model(arch)` with `arch` from `RUNS` (`Unet` by default, `UnetPlusPlus`, `DeepLabV3Plus`), `ENCODER = "resnet34"`, `ENCODER_WEIGHTS = "imagenet"`, `in_channels=1`, `classes=1` |
| `EPOCHS`, `BATCH`, `LR` | 15, 8, 3e-4 (Adam, mixed precision on CUDA) |
| `CROPS_PER_IMAGE` | 4 training samples per train image per epoch, all runs |
| Crop size | 512 (`crop` in `RUNS`) |
| `pos_crop_prob` | 0.5 for `crops_2048`: half the crops are placed so that a random vertex of a random filament polygon falls inside them |
| Loss | BCE with `pos_weight = sqrt((1 - p) / p)`, p = train filament pixel fraction at the run's resolution, plus soft Dice over the batch |
| Augmentation (train only) | random flips, 90-degree rotations, brightness/contrast jitter (same as notebook 02) |
| Prediction | tiles of the crop size, `TILE_OVERLAP = 128`, linear blending weights near tile edges |
| `EVAL_SIZE`, `COMPARE_SIZE` | 2048 (main metrics), 512 (midterm-comparable metrics) |
| Model selection | epoch with the best validation Dice at threshold 0.5 on the 2048 masks |
| Threshold search on val | 0.05 to 0.95 (19 values), separately for each mask size |
| Checkpoints | `OUTPUT_DIR/ckpt_<run>.pt` after every epoch, resumed automatically (`RESUME = 1`) |
| `SPLIT`, `SPLIT_BY_DAY`, `SEED`, `CATEGORY_IDS` | (0.70, 0.15, 0.15), True, 42, None (identical to notebooks 01 and 02) |

**Runtime.** Use a GPU runtime (Runtime -> Change runtime type -> T4 GPU). We have not timed the full notebook on Colab yet. The crop run does four times as many training steps per epoch as notebook 02 and predicts 25 tiles per validation image after every epoch, so expect it to take far longer than the 7.7 minutes of notebook 02. UNet++ needs noticeably more GPU memory and time than U-Net. If it runs out of memory on a T4, add `batch=4` to its entry in `RUNS`. Each run can also be started on its own with the `RUNS` environment variable, for example one run per Colab session. On Colab the config cell mounts Google Drive and writes everything the notebook produces to `MyDrive/DL-Solar-Filaments/outputs`, since files under `/content` are deleted when the session ends. A checkpoint is saved after every epoch, so after a disconnect you rerun the notebook from the top and it carries on where it stopped (section 8).

**Data is NOT included in this repository.** Download it from Kaggle (competition "Solar Filament Segmentation Challenge 2026", slug `filament-segmentation-2026`) after joining the competition and accepting its rules. The download cell asks for the Kaggle token with `getpass` at runtime. Never paste a token into a cell or commit `kaggle.json`.

**Attribution:** the U-Net, UNet++ and DeepLabV3+ models come from the `segmentation_models_pytorch` library and start from ImageNet-pretrained ResNet34 encoder weights downloaded by that library. The data loading, disk cache, crop sampling, tiled prediction, split, loss, training loop and metrics are the team's own code. The split, mask drawing, loss and training loop are carried over from notebook 02.

## 1 - Install, config, imports

In [ ]:
import os, sys, glob, json, random, time, hashlib, platform
if "google.colab" in sys.modules:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

SEED = 42
SPLIT = (0.70, 0.15, 0.15)
SPLIT_BY_DAY = True
CATEGORY_IDS = None         # None = all annotation categories (same as notebooks 01 and 02)

ENCODER = "resnet34"
ENCODER_WEIGHTS = os.environ.get("ENCODER_WEIGHTS", "imagenet")   # "imagenet" = pretrained; "None" = from scratch
if ENCODER_WEIGHTS == "None": ENCODER_WEIGHTS = None
LR = 3e-4
EPOCHS = int(os.environ.get("EPOCHS", 15))
BATCH = int(os.environ.get("BATCH", 8))
CROPS_PER_IMAGE = int(os.environ.get("CROPS_PER_IMAGE", 4))   # training samples per train image per epoch
# DataLoader worker processes. Jupyter on Windows/macOS starts workers with "spawn", which cannot load classes defined
# in a notebook, so the default there is 0. Colab (Linux) uses fork and gets 2.
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", 2 if sys.platform.startswith("linux") else 0))
REUSE_WEIGHTS = os.environ.get("REUSE_WEIGHTS", "0") == "1"   # load OUTPUT_DIR/unet_<run>.pt instead of training, if it exists
RESUME = os.environ.get("RESUME", "1") == "1"                 # continue from OUTPUT_DIR/ckpt_<run>.pt, or skip a finished run

EVAL_SIZE = 2048            # main metrics: masks drawn at the native image size
COMPARE_SIZE = 512          # second metrics: masks drawn at 512, identical to notebook 02
TILE_OVERLAP = 128          # overlap between neighbouring tiles when a full image is predicted
TILE_BATCH = 8              # tiles per forward pass at prediction time
THR_GRID = np.linspace(0.05, 0.95, 19)

# One model is trained per run. Only these settings differ between runs. Optional keys: arch (a segmentation_models_pytorch
# class name, default "Unet"), encoder (default ENCODER) and batch (default BATCH). The two part-1 runs leave them unset,
# so their existing checkpoints still match. If UNet++ runs out of GPU memory, add batch=4 to its entry.
RUNS = {
    "resized_512": dict(work_size=512, crop=512, pos_crop_prob=0.0),   # whole image resized to 512 (midterm setting)
    "crops_2048": dict(work_size=2048, crop=512, pos_crop_prob=0.5),   # 512 crops from the full 2048 image
    # Part 2: same as crops_2048, different decoder
    "unetpp_crops_2048": dict(work_size=2048, crop=512, pos_crop_prob=0.5, arch="UnetPlusPlus"),
    "deeplabv3p_crops_2048": dict(work_size=2048, crop=512, pos_crop_prob=0.5, arch="DeepLabV3Plus"),
}
RUN_NAMES = [r for r in os.environ.get("RUNS", ",".join(RUNS)).split(",") if r]

DATA_DIR = os.environ.get("DATA_DIR", "/content/filament-segmentation-2026")
CACHE_DIR = os.environ.get("CACHE_DIR", "cache")       # decoded images and masks, about 3.5 GB (git-ignored, stays local)
# Checkpoints, weights, result files and figures. On Colab they go to Google Drive by default, because everything under
# /content is deleted when the session ends. Set SAVE_TO_DRIVE=0 or OUTPUT_DIR to opt out.
IN_COLAB = "google.colab" in sys.modules
SAVE_TO_DRIVE = IN_COLAB and os.environ.get("SAVE_TO_DRIVE", "1") == "1" and "OUTPUT_DIR" not in os.environ
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/DL-Solar-Filaments/outputs"
else:
    OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
_ann = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*Annot*train*.json"), recursive=True))
ANN_PATH = _ann[0] if _ann else None
IMG_DIR = os.path.join(os.path.dirname(ANN_PATH), "train_images") if ANN_PATH else None

for r in RUN_NAMES:
    c = RUNS[r]
    assert c["work_size"] % 32 == 0 and c["crop"] % 32 == 0 and c["crop"] <= c["work_size"] <= EVAL_SIZE, (r, c)
    assert TILE_OVERLAP < c["crop"], (r, c)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)   # used only by the split, exactly as in notebooks 01 and 02
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
use_amp = device == "cuda"
print("device:", device, "| runs:", RUN_NAMES, "| annotations:", ANN_PATH)
print("outputs:", os.path.abspath(OUTPUT_DIR), "(Google Drive)" if SAVE_TO_DRIVE else "")
if device == "cpu": print("WARNING: no GPU found, training will be very slow")

## 2 - Get the data (Colab only)
Join the competition on Kaggle first. Paste your Kaggle API token when prompted (or leave it empty to upload a legacy
`kaggle.json`). Skip this cell if the data is already in `DATA_DIR`.

In [ ]:
if "google.colab" in sys.modules and not os.path.exists(DATA_DIR):
    import getpass
    os.system("pip install -q --upgrade kaggle")
    tok = getpass.getpass("Paste Kaggle API token (leave empty to upload kaggle.json instead): ").strip()
    if tok:
        os.environ["KAGGLE_API_TOKEN"] = tok
    else:
        from google.colab import files
        files.upload()
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.system("mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json")
    os.system("kaggle competitions download -c filament-segmentation-2026 -p /content/dl")
    os.system(f"unzip -q -o /content/dl/*.zip -d {DATA_DIR}")
    _ann = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*Annot*train*.json"), recursive=True))
    ANN_PATH = _ann[0]; IMG_DIR = os.path.join(os.path.dirname(ANN_PATH), "train_images")
print(os.path.exists(DATA_DIR), ANN_PATH)

## 3 - Annotations, records and mask drawing

In [ ]:
A = json.load(open(ANN_PATH))
print("images:", len(A["images"]), " annotations:", len(A["annotations"]))
cnt = Counter(a["category_id"] for a in A["annotations"])
for c in sorted(A["categories"], key=lambda c: c["id"]):
    print(f"category id={c['id']}  name={c['name']!r}  -> annotations: {cnt.get(c['id'], 0)}")
print("CATEGORY_IDS =", CATEGORY_IDS, " (None means all categories)")

by_img = defaultdict(list)
for a in A["annotations"]:
    if CATEGORY_IDS is None or a["category_id"] in CATEGORY_IDS:
        by_img[a["image_id"]].append(a)
# Same records and order as load_pairs() in notebooks 01 and 02 (records whose file is missing are skipped).
recs = [r for r in A["images"] if os.path.exists(os.path.join(IMG_DIR, r["file_name"]))]
names = [os.path.splitext(r["file_name"])[0] for r in recs]
n_rle = sum(isinstance(a["segmentation"], dict) for r in recs for a in by_img.get(r["id"], []))
sizes = Counter((r["width"], r["height"]) for r in recs)
print(f"records with an image file: {len(recs)} of {len(A['images'])} | distinct files: {len(set(names))} | "
      f"RLE annotations skipped: {n_rle} | image sizes: {dict(sizes)}")

`polygons()` scales a record's polygons to a square canvas of a given size, the same way `load_pairs()` did in notebooks
01 and 02, and `draw_mask()` rasterizes them with the same PIL call. Every record's mask is drawn once per size and saved
bit-packed in `CACHE_DIR` (about 0.5 MB per record at 2048). Training crops slice their mask out of that file. We first
tried drawing each crop's mask straight from the shifted polygons, but PIL rounds coordinates slightly differently once
a polygon is cut by the crop border, and a test on synthetic data found a few hundred pixels per few hundred crops that disagreed with the full mask.
Slicing a stored mask avoids that, so training and evaluation see exactly the same labels.

In [ ]:
def ann_polygons(a, sx, sy):
    """Polygons of one annotation as (N, 2) arrays, scaled by (sx, sy). RLE masks are skipped, as in notebooks 01 and 02."""
    seg = a["segmentation"]
    if isinstance(seg, dict): return []
    if seg and not isinstance(seg[0], (list, tuple)): seg = [seg]
    out = []
    for poly in seg:
        pts = np.asarray(poly, dtype=float).reshape(-1, 2)
        if len(pts) >= 3: out.append(pts * [sx, sy])
    return out

def polygons(rec, size):
    """All polygons of an image record on a size x size canvas."""
    sx, sy = size / rec["width"], size / rec["height"]
    return [p for a in by_img.get(rec["id"], []) for p in ann_polygons(a, sx, sy)]

def draw_mask(polys, h, w=None, origin=(0, 0)):
    """0/1 mask of an h x w window whose top-left corner sits at origin = (x, y) on the scaled canvas."""
    w = w or h; ox, oy = origin
    m = Image.new("L", (w, h), 0); d = ImageDraw.Draw(m)
    for pts in polys:
        d.polygon([(x - ox, y - oy) for x, y in pts], fill=255, outline=255)
    return (np.asarray(m) > 0).astype(np.uint8)

_mask_caches = {}
def mask_cache(size):
    """Draw every record's mask once at size x size and store it bit-packed (delete CACHE_DIR if the annotations change)."""
    if size in _mask_caches: return _mask_caches[size]
    tag = "all" if CATEGORY_IDS is None else "-".join(map(str, sorted(CATEGORY_IDS)))
    d = os.path.join(CACHE_DIR, f"mask_{size}_{tag}")
    os.makedirs(d, exist_ok=True)
    px_path = os.path.join(d, "filament_pixels.json")
    px = json.load(open(px_path)) if os.path.exists(px_path) else {}
    t0, new = time.time(), 0
    for r in recs:
        out = os.path.join(d, f"{r['id']}.npy")
        if str(r["id"]) in px and os.path.exists(out): continue
        m = draw_mask(polygons(r, size), size)
        np.save(out, np.packbits(m, axis=1)); px[str(r["id"])] = int(m.sum()); new += 1
    if new:
        with open(px_path, "w") as f: json.dump(px, f)
    print(f"mask cache {d}: {len(recs)} records ({new} drawn now, {time.time() - t0:.0f}s)")
    _mask_caches[size] = {"dir": d, "size": size, "pixels": np.array([px[str(r["id"])] for r in recs])}
    return _mask_caches[size]

def read_mask(mc, rec, y0=0, x0=0, h=None, w=None):
    """A window of a cached mask (uint8 0/1). Only the rows of the window are read from disk."""
    a = np.load(os.path.join(mc["dir"], f"{rec['id']}.npy"), mmap_mode="r")
    h, w = h or mc["size"], w or mc["size"]
    return np.unpackbits(np.asarray(a[y0:y0 + h]), axis=1, count=mc["size"])[:, x0:x0 + w]

def filament_pixels(size):
    """Filament pixel count of every record (in recs order) for masks drawn at size x size."""
    return mask_cache(size)["pixels"]

### Repeated file names
The midterm found about 300 file names that appear in more than one image record. We kept every record as its own
example there, and we keep doing that here so the two notebooks stay comparable. This cell only measures how much the
masks of records that share a file overlap, which tells us whether the repeats are copies or separate partial labelings of the same picture.

In [ ]:
groups = defaultdict(list)
for i, r in enumerate(recs): groups[r["file_name"]].append(i)
dup_groups = [ix for ix in groups.values() if len(ix) > 1]
kinds, ious = Counter(), []
for ix in dup_groups:
    ms = [draw_mask(polygons(recs[i], COMPARE_SIZE), COMPARE_SIZE).astype(bool) for i in ix]
    inter, union = np.logical_and.reduce(ms).sum(), np.logical_or.reduce(ms).sum()
    iou = inter / union if union else 1.0
    ious.append(iou)
    kinds["identical masks" if iou == 1 else "no overlap at all" if inter == 0 else "partial overlap"] += 1
print(f"file names with more than one record: {len(dup_groups)} ({sum(len(ix) - 1 for ix in dup_groups)} extra records)")
print("masks of records sharing a file (512 x 512):", dict(kinds))
if ious: print(f"IoU between the masks within a group: median {np.median(ious):.3f}, min {np.min(ious):.3f}")

## 4 - Split by day
The code and seed are identical to notebooks 01 and 02, so all three notebooks use the same train, validation and test images.
The split is also written to `OUTPUT_DIR/split_seed42.json` with a short fingerprint, which makes it easy to confirm later.

In [ ]:
n = len(recs)
days = np.array([nm[:8] for nm in names])
if SPLIT_BY_DAY and all(d.isdigit() for d in days):
    ud = rng.permutation(np.unique(days))
    cum = np.cumsum([(days == d).sum() for d in ud])
    k1, k2 = np.searchsorted(cum, SPLIT[0] * n) + 1, np.searchsorted(cum, (SPLIT[0] + SPLIT[1]) * n) + 1
    which = {d: 0 for d in ud[:k1]} | {d: 1 for d in ud[k1:k2]} | {d: 2 for d in ud[k2:]}
    grp = np.array([which[d] for d in days])
    idx_tr, idx_va, idx_te = (np.flatnonzero(grp == g) for g in range(3))
else:
    perm = rng.permutation(n)
    n_tr, n_va = int(SPLIT[0] * n), int(SPLIT[1] * n)
    idx_tr, idx_va, idx_te = perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]
print(f"train={len(idx_tr)}  val={len(idx_va)}  test={len(idx_te)} images")

split = {s: [names[i] for i in idx] for s, idx in (("train", idx_tr), ("val", idx_va), ("test", idx_te))}
split_fp = hashlib.md5(json.dumps(split).encode()).hexdigest()[:12]
with open(os.path.join(OUTPUT_DIR, f"split_seed{SEED}.json"), "w") as f:
    json.dump({"seed": SEED, "fingerprint": split_fp, **split}, f)
print("split fingerprint:", split_fp)

## Dataset statistics and leakage checks
Filament pixel fractions are reported for masks drawn at 2048 and at 512. For the real data, notebook 02 reported train
0.4590%, val 0.5565% and test 0.4223% at 512, so the 512 column should show the same values. The two leakage assertions are the same as in notebooks 01 and 02.

In [ ]:
_ann_per_img = Counter(a["image_id"] for a in A["annotations"] if CATEGORY_IDS is None or a["category_id"] in CATEGORY_IDS)
_fmt_day = lambda d: f"{d[:4]}-{d[4:6]}-{d[6:8]}" if d.isdigit() else d
px_eval, px_cmp = filament_pixels(EVAL_SIZE), filament_pixels(COMPARE_SIZE)
split_days, split_names = {}, {}
for split_name, idx in (("train", idx_tr), ("val", idx_va), ("test", idx_te)):
    d = sorted(set(days[idx]))
    split_days[split_name], split_names[split_name] = set(d), {names[i] for i in idx}
    f_eval = px_eval[idx].sum() / (len(idx) * EVAL_SIZE ** 2)
    f_cmp = px_cmp[idx].sum() / (len(idx) * COMPARE_SIZE ** 2)
    n_ann = sum(_ann_per_img.get(recs[i]["id"], 0) for i in idx)
    print(f"{split_name:5s}: images={len(idx):5d}  days={len(d):4d}  first={_fmt_day(d[0]) if d else '-'}  "
          f"last={_fmt_day(d[-1]) if d else '-'}  filament px @{EVAL_SIZE}={f_eval:.4%}  @{COMPARE_SIZE}={f_cmp:.4%}  "
          f"annotations={n_ann}")

_pairs = [("train", "val"), ("train", "test"), ("val", "test")]
day_overlap = {f"{a}/{b}": sorted(split_days[a] & split_days[b]) for a, b in _pairs}
name_overlap = {f"{a}/{b}": sorted(split_names[a] & split_names[b]) for a, b in _pairs}
n_day_overlap = sum(len(v) for v in day_overlap.values())
n_name_overlap = sum(len(v) for v in name_overlap.values())
print(f"Check 1 - days in more than one split: {n_day_overlap}", {k: len(v) for k, v in day_overlap.items()})
print(f"Check 2 - file names in more than one split: {n_name_overlap}", {k: len(v) for k, v in name_overlap.items()})
assert n_name_overlap == 0, f"file names shared between splits: {name_overlap}"
if SPLIT_BY_DAY:
    assert n_day_overlap == 0, f"days shared between splits (day split did not run?): {day_overlap}"
else:
    print("SPLIT_BY_DAY is False, so check 1 is reported but not asserted")
print("Leakage checks passed")

## 5 - Disk cache
Every image file is decoded once at the run's working size (2048 or 512) and saved as an 8-bit `.npy` file. Training then
opens the file as a memory map and reads only the rows and columns of the crop it needs. The 2048 cache takes about 4 MB
per file. Images are decoded exactly as in `load_pairs()` (grayscale, `draft` + bilinear resize), and each one is scaled to 0-1 with the
minimum and maximum of the whole image. A crop therefore gets the same pixel values it would have had if the full image had been loaded.

In [ ]:
def build_cache(size):
    d = os.path.join(CACHE_DIR, f"img_{size}")
    os.makedirs(d, exist_ok=True)
    mm_path = os.path.join(d, "minmax.json")
    minmax = json.load(open(mm_path)) if os.path.exists(mm_path) else {}
    t0, new = time.time(), 0
    for fn in sorted({r["file_name"] for r in recs}):
        out = os.path.join(d, os.path.splitext(fn)[0] + ".npy")
        if fn in minmax and os.path.exists(out): continue
        im = Image.open(os.path.join(IMG_DIR, fn)); im.draft("L", (size, size))
        a = np.asarray(im.convert("L").resize((size, size), Image.BILINEAR), dtype=np.uint8)
        np.save(out, a); minmax[fn] = [int(a.min()), int(a.max())]; new += 1
    with open(mm_path, "w") as f: json.dump(minmax, f)
    print(f"cache {d}: {len(minmax)} files ({new} decoded now, {time.time() - t0:.0f}s)")
    return {"dir": d, "minmax": minmax, "size": size}

def read_image(cache, rec, y0=0, x0=0, h=None, w=None):
    """A window of a cached image, scaled to 0-1 with the whole image's min and max (only the window is read from disk)."""
    a = np.load(os.path.join(cache["dir"], os.path.splitext(rec["file_name"])[0] + ".npy"), mmap_mode="r")
    h, w = h or a.shape[0], w or a.shape[1]
    lo, hi = cache["minmax"][rec["file_name"]]
    return (np.asarray(a[y0:y0 + h, x0:x0 + w], dtype=np.float32) - lo) / (hi - lo + 1e-8)

## 6 - Datasets: random crops for training, whole images for evaluation
A training sample is a `crop` x `crop` window of a training image at the run's working size. With probability
`pos_crop_prob` the window is placed so that a random vertex of a random filament polygon lands at a random spot inside
it, and the crop's mask is sliced from the mask cache. Otherwise the position is uniform. Filaments cover well under 1% of the image, so without this rule many crops would
show nothing but background or empty space around the disc. When `crop` equals the working size (the `resized_512` run)
the crop is simply the whole image. Augmentation is the same as in notebook 02 and touches training samples only.
Evaluation samples are whole images together with their ground-truth masks at 2048 and at 512.

In [ ]:
def seed_worker(_):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

class CropTrainData(Dataset):
    def __init__(self, idxs, cache, crop, pos_prob, n_per_image, augment=True):
        self.items = np.repeat(np.asarray(idxs), n_per_image)
        self.cache, self.size, self.crop, self.pos_prob, self.augment = cache, cache["size"], crop, pos_prob, augment
        self.masks = mask_cache(self.size)
        self.polys = {int(i): polygons(recs[i], self.size) for i in set(self.items.tolist())}
    def __len__(self): return len(self.items)
    def origin(self, polys):
        S, c = self.size, self.crop
        if c >= S: return 0, 0
        if polys and random.random() < self.pos_prob:
            pts = polys[random.randrange(len(polys))]
            vx, vy = pts[random.randrange(len(pts))]
            ox, oy = int(vx - random.randrange(c)), int(vy - random.randrange(c))
        else:
            ox, oy = random.randrange(S - c + 1), random.randrange(S - c + 1)
        return min(max(ox, 0), S - c), min(max(oy, 0), S - c)
    def __getitem__(self, k):
        i = int(self.items[k]); polys = self.polys[i]
        ox, oy = self.origin(polys)
        x = torch.from_numpy(read_image(self.cache, recs[i], oy, ox, self.crop, self.crop))[None]   # (1, c, c)
        y = torch.from_numpy(read_mask(self.masks, recs[i], oy, ox, self.crop, self.crop).astype(np.float32))[None]
        if self.augment:
            if random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
            if random.random() < 0.5: x, y = x.flip(-2), y.flip(-2)
            r = random.randint(0, 3)
            if r: x, y = torch.rot90(x, r, (-2, -1)), torch.rot90(y, r, (-2, -1))
            x = (x * random.uniform(0.85, 1.15) + random.uniform(-0.05, 0.05)).clamp(0, 1)
        return x.contiguous(), y.contiguous()

class FullImageData(Dataset):
    def __init__(self, idxs, cache):
        self.idxs, self.cache = np.asarray(idxs), cache
        self.m_eval, self.m_cmp = mask_cache(EVAL_SIZE), mask_cache(COMPARE_SIZE)
    def __len__(self): return len(self.idxs)
    def __getitem__(self, k):
        rec = recs[int(self.idxs[k])]
        x = torch.from_numpy(read_image(self.cache, rec))[None]
        y_eval = torch.from_numpy(read_mask(self.m_eval, rec))
        y_cmp = torch.from_numpy(read_mask(self.m_cmp, rec))
        return x, y_eval, y_cmp, k

## 7 - Tiled prediction and metrics
`predict_image()` slides a `crop` x `crop` window over the image with `TILE_OVERLAP` pixels of overlap. For 2048 and 512 that gives
start positions 0, 384, 768, 1152 and 1536 along each axis, so 25 tiles. Where tiles overlap, their probabilities are averaged with weights that fall off
linearly over the last `TILE_OVERLAP` pixels before a tile edge, because predictions near a tile edge have the least context.
If the crop covers the whole image, one forward pass is enough.

The probability map is then resized to each evaluation size: area averaging when it is larger, bilinear upsampling when
it is smaller. Pooled TP, FP and FN are counted per image and per threshold, so the threshold search on validation needs only one
pass over the images. `annotation_recall()` measures, for every annotated filament, what fraction of its pixels the
thresholded prediction covers, together with an approximate width (2 x area / perimeter, in 2048-scale pixels).

In [ ]:
def _ramp(tile, overlap):
    i = torch.arange(tile, dtype=torch.float32)
    return torch.minimum(torch.minimum(i + 1, tile - i), torch.tensor(float(overlap))) / overlap

@torch.no_grad()
def predict_image(model, x, tile, overlap=TILE_OVERLAP):
    """Filament probability map (S, S) for one image tensor x of shape (1, 1, S, S) on the device."""
    S = x.shape[-1]
    if tile >= S:
        with torch.autocast(device_type=device, enabled=use_amp):
            return torch.sigmoid(model(x).float())[0, 0]
    starts = list(range(0, S - tile, tile - overlap)) + [S - tile]
    r = _ramp(tile, overlap).to(x.device); w = r[:, None] * r[None, :]
    acc = torch.zeros(S, S, device=x.device); wsum = torch.zeros(S, S, device=x.device)
    coords = [(y0, x0) for y0 in starts for x0 in starts]
    for b in range(0, len(coords), TILE_BATCH):
        cs = coords[b:b + TILE_BATCH]
        batch = torch.cat([x[..., y0:y0 + tile, x0:x0 + tile] for y0, x0 in cs])
        with torch.autocast(device_type=device, enabled=use_amp):
            p = torch.sigmoid(model(batch).float())[:, 0]
        for (y0, x0), pi in zip(cs, p):
            acc[y0:y0 + tile, x0:x0 + tile] += pi * w
            wsum[y0:y0 + tile, x0:x0 + tile] += w
    return acc / wsum

def to_size(p, size):
    if p.shape[-1] == size: return p
    if p.shape[-1] > size: return F.interpolate(p[None, None], size=(size, size), mode="area")[0, 0]
    return F.interpolate(p[None, None], size=(size, size), mode="bilinear", align_corners=False)[0, 0]

def counts(p, gt, thrs):
    """(len(thrs), 3) array of TP, FP, FN for prediction p >= t."""
    gt = gt.bool(); n_pos = gt.sum()
    out = []
    for t in thrs:
        pr = p >= float(t); tp = (pr & gt).sum()
        out.append(torch.stack([tp, pr.sum() - tp, n_pos - tp]))
    return torch.stack(out).cpu().numpy().astype(np.int64)

def scores(tp, fp, fn, eps=1e-8):
    return {"IoU": tp / (tp + fp + fn + eps), "Dice": 2 * tp / (2 * tp + fp + fn + eps),
            "Precision": tp / (tp + fp + eps), "Recall": tp / (tp + fn + eps)}

def annotation_recall(pred, i):
    """Per-annotation recall on image record i. pred is a bool numpy array at EVAL_SIZE."""
    rec = recs[i]; sx, sy = EVAL_SIZE / rec["width"], EVAL_SIZE / rec["height"]
    out = []
    for a in by_img.get(rec["id"], []):
        polys = ann_polygons(a, sx, sy)
        if not polys: continue
        allp = np.concatenate(polys)
        x0, y0 = np.maximum(np.floor(allp.min(0)).astype(int) - 1, 0)
        x1, y1 = np.minimum(np.ceil(allp.max(0)).astype(int) + 2, EVAL_SIZE)
        m = draw_mask(polys, y1 - y0, x1 - x0, origin=(x0, y0)).astype(bool)
        area = int(m.sum())
        if area == 0: continue
        perim = sum(np.linalg.norm(np.diff(np.vstack([p, p[:1]]), axis=0), axis=1).sum() for p in polys)
        out.append({"record": int(i), "ann_id": a["id"], "width_px": 2 * area / max(perim, 1e-6),
                    "area_px": area, "recall": float(pred[y0:y1, x0:x1][m].mean())})
    return out

@torch.no_grad()
def evaluate(model, cfg, cache, idxs, thr_eval, thr_cmp, ann_thr=None):
    """Predict every image in idxs and count TP/FP/FN per image at both mask sizes."""
    model.eval()
    dl = DataLoader(FullImageData(idxs, cache), batch_size=1, num_workers=NUM_WORKERS, worker_init_fn=seed_worker)
    c_eval = np.zeros((len(idxs), len(thr_eval), 3), np.int64); c_cmp = np.zeros((len(idxs), len(thr_cmp), 3), np.int64)
    ann, t0 = [], time.time()
    for x, y_eval, y_cmp, k in dl:
        k = int(k)
        p = predict_image(model, x.to(device), cfg["crop"])
        p_eval = to_size(p, EVAL_SIZE)
        c_eval[k] = counts(p_eval, y_eval[0].to(device), thr_eval)
        c_cmp[k] = counts(to_size(p, COMPARE_SIZE), y_cmp[0].to(device), thr_cmp)
        if ann_thr is not None:
            ann += annotation_recall((p_eval >= float(ann_thr)).cpu().numpy(), int(idxs[k]))
    return {"eval": c_eval, "cmp": c_cmp, "ann": ann, "sec_per_image": (time.time() - t0) / max(len(idxs), 1)}

## 8 - Training
Loss, optimizer and the training loop are the same as in notebook 02. The BCE `pos_weight` comes from the filament
fraction of the training masks at the run's working size. After every epoch the model predicts all validation images in
full (tiled for the crop run) and is scored with Dice at threshold 0.5 against the 2048 masks. The best epoch's
weights are kept and saved to `OUTPUT_DIR/unet_<run>.pt`. The file keeps that name for every architecture, so weights
saved by earlier runs still load.

A checkpoint (`OUTPUT_DIR/ckpt_<run>.pt`) is written at the end of every epoch. It holds the model, the optimizer and
grad-scaler state, the best weights so far, the history and every random number generator state. If Colab disconnects,
run the notebook again from the top. Finished runs are skipped, and an unfinished run picks up after its last completed
epoch, drawing the same crops it would have drawn without the interruption. A checkpoint only resumes a run with the same
settings (crop, sampling, architecture, batch, learning rate, encoder, seed, split). For anything else the notebook stops and asks for a new
run name, which keeps the part 3 grid from mixing runs. `EPOCHS` may be raised for an unfinished run. Each checkpoint is about 400 MB
while its run trains. Once the run ends it shrinks to the history alone, and the weights stay in `unet_<run>.pt`. `RESUME=0` ignores
existing checkpoints and starts fresh.

In [ ]:
def make_loss(pos_frac):
    pos_weight = torch.tensor(float(np.sqrt((1 - pos_frac) / max(pos_frac, 1e-6))), device=device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    def loss_fn(logits, y):
        p = torch.sigmoid(logits)
        return bce(logits, y) + 1 - (2 * (p * y).sum() + 1.0) / (p.sum() + y.sum() + 1.0)
    return loss_fn, pos_weight.item()

def new_model(cfg):
    """The run's architecture (cfg["arch"], default Unet) with the run's encoder, one input channel and one output class."""
    return smp.create_model(cfg.get("arch", "Unet"), encoder_name=cfg.get("encoder", ENCODER),
                            encoder_weights=ENCODER_WEIGHTS, in_channels=1, classes=1).to(device)

def save_atomic(obj, path):
    """Write to a temporary file first, so a disconnect during the write never leaves a half-written file behind."""
    tmp = path + ".tmp"
    torch.save(obj, tmp); os.replace(tmp, path)

def run_signature(cfg):
    """Settings a checkpoint must match before training is resumed from it (EPOCHS may grow)."""
    return {**cfg, "crops_per_image": CROPS_PER_IMAGE, "batch": cfg.get("batch", BATCH), "lr": LR,
            "encoder": cfg.get("encoder", ENCODER),
            "encoder_weights": ENCODER_WEIGHTS, "seed": SEED, "split_fingerprint": split_fp, "category_ids": CATEGORY_IDS}

def rng_state(g):
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None, "loader": g.get_state()}

def set_rng_state(st, g):
    random.setstate(st["python"]); np.random.set_state(st["numpy"]); torch.set_rng_state(st["torch"])
    if st["cuda"] is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(st["cuda"])
    g.set_state(st["loader"])

def train_run(name, cfg):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    cache = build_cache(cfg["work_size"])
    S = cfg["work_size"]
    pos_frac = filament_pixels(S)[idx_tr].sum() / (len(idx_tr) * S * S)
    loss_fn, pw = make_loss(pos_frac)
    model = new_model(cfg)
    info = {"pos_frac": float(pos_frac), "pos_weight": pw, "hist": {"train_loss": [], "val_dice": [], "val_dice_cmp": [],
            "epoch_sec": []}, "best_epoch": None, "train_sec": 0.0, "reused_weights": False, "resumed_after_epoch": []}
    wpath = os.path.join(OUTPUT_DIR, f"unet_{name}.pt")
    cpath = os.path.join(OUTPUT_DIR, f"ckpt_{name}.pt")
    if REUSE_WEIGHTS and os.path.exists(wpath):
        model.load_state_dict(torch.load(wpath, map_location=device)); info["reused_weights"] = True
        print(f"[{name}] loaded {wpath}, training skipped")
        return model, cache, info

    sig = run_signature(cfg)
    ck = torch.load(cpath, map_location="cpu", weights_only=False) if RESUME and os.path.exists(cpath) else None
    if ck is not None and ck["signature"] != sig:
        diff = {k: (ck["signature"].get(k), v) for k, v in sig.items() if ck["signature"].get(k) != v}
        raise RuntimeError(f"[{name}] {cpath} was saved with different settings {diff} (saved, now). "
                           "Give the run a new name in RUNS, or delete the checkpoint to start over.")
    if ck is not None and ck["done"]:
        if ck["epochs_done"] < EPOCHS:
            print(f"[{name}] this run already finished after {ck['epochs_done']} epochs, so EPOCHS={EPOCHS} is ignored. "
                  "Delete the checkpoint and weights to retrain it.")
        model.load_state_dict(torch.load(wpath, map_location=device)); info = ck["info"]
        print(f"[{name}] finished run found, loaded {wpath}, training skipped")
        return model, cache, info

    ds = CropTrainData(idx_tr, cache, cfg["crop"], cfg["pos_crop_prob"], CROPS_PER_IMAGE)
    g = torch.Generator(); g.manual_seed(SEED)
    # persistent_workers stays off: workers are re-seeded from g every epoch, so a resumed run draws the same crops
    # it would have drawn without the interruption.
    bs = cfg.get("batch", BATCH)
    train_dl = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, worker_init_fn=seed_worker,
                          generator=g, drop_last=len(ds) > bs, pin_memory=device == "cuda")
    print(f"[{name}] {cfg.get('arch', 'Unet')} / {cfg.get('encoder', ENCODER)}, work size {S}, crop {cfg['crop']}, "
          f"batch {bs}, {len(ds)} samples / {len(train_dl)} batches per epoch, "
          f"train filament fraction {pos_frac:.4%} -> pos_weight {pw:.1f}, parameters {sum(p.numel() for p in model.parameters()):,}")
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best_dice, best_state, start_ep = -1.0, None, 0
    if ck is not None:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        best_dice, best_state, info, start_ep = ck["best_dice"], ck["best_state"], ck["info"], ck["epochs_done"]
        info["resumed_after_epoch"].append(start_ep)
        set_rng_state(ck["rng"], g)
        print(f"[{name}] resumed from {cpath} after epoch {start_ep} (best val Dice so far {best_dice:.4f})")
    for ep in range(start_ep, EPOCHS):
        model.train(); t0 = time.time(); run = 0.0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with torch.autocast(device_type=device, enabled=use_amp):
                logits = model(x)
            loss = loss_fn(logits.float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += loss.item() * len(x)
        tr_loss = run / (len(train_dl) * bs if train_dl.drop_last else len(ds))
        ev = evaluate(model, cfg, cache, idx_va, [0.5], [0.5])
        vd = float(scores(*ev["eval"].sum(0)[0])["Dice"]); vd_c = float(scores(*ev["cmp"].sum(0)[0])["Dice"])
        h = info["hist"]; h["train_loss"].append(tr_loss); h["val_dice"].append(vd); h["val_dice_cmp"].append(vd_c)
        h["epoch_sec"].append(time.time() - t0)
        if vd > best_dice:
            best_dice, info["best_epoch"] = vd, ep + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        t_save = time.time()
        save_atomic({"signature": sig, "done": False, "epochs_done": ep + 1, "model": model.state_dict(),
                     "opt": opt.state_dict(), "scaler": scaler.state_dict(), "best_dice": best_dice,
                     "best_state": best_state, "info": info, "rng": rng_state(g)}, cpath)
        print(f"[{name}] epoch {ep+1:2d}/{EPOCHS}  train loss {tr_loss:.4f}  val Dice@{EVAL_SIZE} {vd:.4f}  "
              f"@{COMPARE_SIZE} {vd_c:.4f}  best {best_dice:.4f}  ({t_save-t0:.0f}s + {time.time()-t_save:.0f}s checkpoint)")
    info["train_sec"] = float(sum(info["hist"]["epoch_sec"]))
    model.load_state_dict(best_state)
    save_atomic(best_state, wpath)
    # The finished run keeps only its best weights (unet_<run>.pt) and a small checkpoint with the history,
    # so a rerun skips it and Drive does not fill up with optimizer states.
    save_atomic({"signature": sig, "done": True, "epochs_done": EPOCHS, "info": info}, cpath)
    return model, cache, info

## 9 - Threshold tuning and the single test pass
For each mask size the threshold with the highest validation Dice is chosen from `THR_GRID`. The test images are then
predicted once and scored at those two thresholds only. Results go to `OUTPUT_DIR/results_<run>.json`, together with
the configuration, training history, per-image test counts and per-annotation recall.

In [ ]:
def final_eval(name, model, cfg, cache, info):
    va = evaluate(model, cfg, cache, idx_va, THR_GRID, THR_GRID)
    pv_eval, pv_cmp = va["eval"].sum(0), va["cmp"].sum(0)
    j_eval = int(np.argmax([scores(*c)["Dice"] for c in pv_eval]))   # first best, like max() in notebooks 01 and 02
    j_cmp = int(np.argmax([scores(*c)["Dice"] for c in pv_cmp]))
    thr_eval, thr_cmp = float(THR_GRID[j_eval]), float(THR_GRID[j_cmp])
    te = evaluate(model, cfg, cache, idx_te, [thr_eval], [thr_cmp], ann_thr=thr_eval)
    f = lambda c: {k: float(v) for k, v in scores(*c).items()}
    res = {
        "run": name, "config": {**cfg, "arch": cfg.get("arch", "Unet"), "crops_per_image": CROPS_PER_IMAGE,
                                "epochs": EPOCHS, "epochs_trained": len(info["hist"]["train_loss"]) or None,
                                "batch": cfg.get("batch", BATCH), "lr": LR, "encoder": cfg.get("encoder", ENCODER),
                                "encoder_weights": ENCODER_WEIGHTS, "tile_overlap": TILE_OVERLAP,
                                "seed": SEED, "split_fingerprint": split_fp},
        "params": int(sum(p.numel() for p in model.parameters())),
        "threshold": {str(EVAL_SIZE): thr_eval, str(COMPARE_SIZE): thr_cmp},
        "val": {str(EVAL_SIZE): f(pv_eval[j_eval]), str(COMPARE_SIZE): f(pv_cmp[j_cmp])},
        "test": {str(EVAL_SIZE): f(te["eval"].sum(0)[0]), str(COMPARE_SIZE): f(te["cmp"].sum(0)[0])},
        "test_per_image": [{"name": names[i], "tp_fp_fn": te["eval"][k, 0].tolist()} for k, i in enumerate(idx_te)],
        "test_annotations": te["ann"], "test_sec_per_image": te["sec_per_image"],
        **{k: v for k, v in info.items()},
        "env": {"python": platform.python_version(), "torch": torch.__version__, "smp": smp.__version__,
                "device": torch.cuda.get_device_name(0) if device == "cuda" else device},
    }
    with open(os.path.join(OUTPUT_DIR, f"results_{name}.json"), "w") as fh:
        json.dump(res, fh, indent=1)
    print(f"[{name}] thresholds tuned on val: @{EVAL_SIZE} {thr_eval:.2f}, @{COMPARE_SIZE} {thr_cmp:.2f}")
    for s in (str(EVAL_SIZE), str(COMPARE_SIZE)):
        print(f"[{name}] val  @{s}:", {k: round(v, 4) for k, v in res["val"][s].items()})
        print(f"[{name}] test @{s}:", {k: round(v, 4) for k, v in res["test"][s].items()})
    return res

## 10 - Run all experiments

In [ ]:
results, models = {}, {}
for name in RUN_NAMES:
    cfg = RUNS[name]
    model, cache, info = train_run(name, cfg)
    results[name] = final_eval(name, model, cfg, cache, info)
    models[name] = (model, cfg, cache)

## 11 - Comparison
The midterm row is copied from Table III of the midterm report (notebook 02, 512 x 512 masks, test split). It was trained
on the whole image once per epoch, so it got a quarter of the training steps of the runs here. The three
`crops_2048` rows (U-Net, UNet++ and DeepLabV3+) share every setting except the decoder, which is the part 2 comparison.

In [ ]:
rows = {"U-Net 512, midterm (notebook 02)": {"model": "Unet", "params (M)": 24.4,
                                               f"IoU@{COMPARE_SIZE}": 0.5331, f"Dice@{COMPARE_SIZE}": 0.6955,
                                               f"Precision@{COMPARE_SIZE}": 0.6824, f"Recall@{COMPARE_SIZE}": 0.7090}}
for name, r in results.items():
    row = {"model": r["config"]["arch"], "params (M)": round(r["params"] / 1e6, 1)}
    for s in (str(EVAL_SIZE), str(COMPARE_SIZE)):
        row.update({f"{k}@{s}": v for k, v in r["test"][s].items()})
        row[f"thr@{s}"] = r["threshold"][s]
    row.update({"best epoch": r["best_epoch"], "train min": round(r["train_sec"] / 60, 1),
                "test s/image": round(r["test_sec_per_image"], 2)})
    rows[name] = row
table = pd.DataFrame.from_dict(rows, orient="index")   # keeps numeric columns numeric next to the text column
display(table.round(4))
table.to_csv(os.path.join(OUTPUT_DIR, "comparison_test.csv"))

if any(r["hist"]["val_dice"] for r in results.values()):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    for name, r in results.items():
        if not r["hist"]["val_dice"]: continue
        ep = range(1, len(r["hist"]["val_dice"]) + 1)
        ax[0].plot(ep, r["hist"]["train_loss"], label=name); ax[1].plot(ep, r["hist"]["val_dice"], label=name)
    ax[0].set_title("train loss"); ax[1].set_title(f"validation Dice (thr 0.5, {EVAL_SIZE} masks)")
    for a in ax: a.set_xlabel("epoch"); a.legend()
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "curves.png"), dpi=150); plt.show()

### Recall by filament width
Each annotated test filament gets an approximate width (2 x area / perimeter at 2048 scale). The filaments are split into four groups at the
width quartiles, and for every run we average the share of each filament's pixels that the prediction covers. A width of
4 pixels at 2048 is a single pixel at 512. If full resolution helps where we expect it to, the gap between the runs should be largest in the thinnest group.

In [ ]:
ann_tables = {n: pd.DataFrame(r["test_annotations"]) for n, r in results.items() if r["test_annotations"]}
if ann_tables:
    ref = next(iter(ann_tables.values()))
    edges = np.quantile(ref["width_px"], [0, 0.25, 0.5, 0.75, 1.0])
    edges[0], edges[-1] = 0, np.inf
    labels = [f"{edges[j]:.1f}-{edges[j+1]:.1f} px" if np.isfinite(edges[j + 1]) else f">{edges[j]:.1f} px" for j in range(4)]
    by_width = {}
    for name, t in ann_tables.items():
        g = t.groupby(pd.cut(t["width_px"], edges, labels=labels, right=False), observed=False)["recall"]
        by_width[name] = g.mean()
    wt = pd.DataFrame(by_width)
    wt.insert(0, "filaments", ref.groupby(pd.cut(ref["width_px"], edges, labels=labels, right=False), observed=False).size())
    wt.index.name = f"width at {EVAL_SIZE} px"
    display(wt.round(3))
    wt.to_csv(os.path.join(OUTPUT_DIR, "recall_by_width.csv"))

## 12 - Qualitative results
Three test images are picked by their per-image Dice for the `crops_2048` U-Net (worst, median, best), or for the last
run in `RUN_NAMES` if that run was left out. Every trained model is shown on the same three images. For each one the view zooms
into a 512 x 512 window at full resolution, centred on that image's thinnest annotated filament. The overlay colours
are green for correct filament pixels, red for false positives and blue for missed filament pixels.

In [ ]:
def overlay(img, pred, gt):
    rgb = np.repeat(img[..., None], 3, -1) * 0.75
    rgb[pred & gt] = [0.1, 0.9, 0.1]; rgb[pred & ~gt] = [0.95, 0.15, 0.1]; rgb[~pred & gt] = [0.15, 0.4, 1.0]
    return rgb

if results:
    ref_name = "crops_2048" if "crops_2048" in results else list(results)[-1]
    per = results[ref_name]["test_per_image"]
    d = np.array([scores(*p["tp_fp_fn"])["Dice"] for p in per])
    order = np.argsort(d)
    picks = [("worst", order[0]), ("median", order[len(order) // 2]), ("best", order[-1])]
    ann_ref = pd.DataFrame(results[ref_name]["test_annotations"])
    W = 512
    fig, ax = plt.subplots(len(picks), 3 + len(models), figsize=(3.3 * (3 + len(models)), 3.4 * len(picks)), squeeze=False)
    for row, (tag, k) in enumerate(picks):
        i = int(idx_te[k]); rec = recs[i]
        im = Image.open(os.path.join(IMG_DIR, rec["file_name"])); im.draft("L", (EVAL_SIZE, EVAL_SIZE))
        img = np.asarray(im.convert("L").resize((EVAL_SIZE, EVAL_SIZE), Image.BILINEAR), dtype=np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        gt = draw_mask(polygons(rec, EVAL_SIZE), EVAL_SIZE).astype(bool)
        a_i = ann_ref[ann_ref["record"] == i] if len(ann_ref) else ann_ref
        if len(a_i):
            ys, xs = np.nonzero(draw_mask([p for a in by_img[rec["id"]] if a["id"] == a_i.loc[a_i["width_px"].idxmin(), "ann_id"]
                                           for p in ann_polygons(a, EVAL_SIZE / rec["width"], EVAL_SIZE / rec["height"])], EVAL_SIZE))
        else:
            ys, xs = np.nonzero(gt)
        cy, cx = (int(ys.mean()), int(xs.mean())) if len(ys) else (EVAL_SIZE // 2, EVAL_SIZE // 2)
        y0, x0 = int(np.clip(cy - W // 2, 0, EVAL_SIZE - W)), int(np.clip(cx - W // 2, 0, EVAL_SIZE - W))
        win = np.s_[y0:y0 + W, x0:x0 + W]
        ax[row, 0].imshow(img, cmap="gray"); ax[row, 0].add_patch(plt.Rectangle((x0, y0), W, W, fill=False, ec="yellow", lw=1.5))
        ax[row, 0].set_title(f"{tag}: {names[i]}\nDice {d[k]:.3f} ({ref_name})", fontsize=9)
        ax[row, 1].imshow(img[win], cmap="gray"); ax[row, 1].set_title("image (zoom)", fontsize=9)
        ax[row, 2].imshow(gt[win], cmap="gray"); ax[row, 2].set_title("ground truth", fontsize=9)
        for j, (name, (m, cfg, cache)) in enumerate(models.items()):
            x = torch.from_numpy(read_image(cache, rec))[None, None].to(device)
            m.eval(); p = to_size(predict_image(m, x, cfg["crop"]), EVAL_SIZE).cpu().numpy()
            pred = p >= results[name]["threshold"][str(EVAL_SIZE)]
            ax[row, 3 + j].imshow(overlay(img[win], pred[win], gt[win])); ax[row, 3 + j].set_title(name, fontsize=9)
        for a in ax[row]: a.axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "qualitative_test.png"), dpi=150); plt.show()

## 13 - Make sure the files reached Google Drive
Colab copies files to Drive in the background. This cell lists what is in `OUTPUT_DIR` and then flushes any pending
writes. It unmounts Drive while doing so, so run the config cell again before running other cells after this one.

In [ ]:
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print(f"{os.path.getsize(os.path.join(OUTPUT_DIR, fn)) / 2**20:9.1f} MB  {fn}")
if SAVE_TO_DRIVE:
    drive.flush_and_unmount()
    print("Drive flushed and unmounted. Files are in MyDrive/DL-Solar-Filaments/outputs.")

## Notes for the final report
The result files in `OUTPUT_DIR` hold everything the report needs: `comparison_test.csv` for the main table,
`recall_by_width.csv` for the thin-filament question, `curves.png` and `qualitative_test.png` as figures, and one
`results_<run>.json` per run with the configuration and the tuned thresholds. On Colab they are all in the Drive folder. Part 2 of the plan (a second
architecture) is the `unetpp_crops_2048` and `deeplabv3p_crops_2048` runs. Part 3 (the hyperparameter comparison) can
reuse this notebook the same way, by adding entries to `RUNS`. Architecture, encoder and batch size are already read
from the run's `cfg`. The loss and learning rate still come from the global settings and would have to be moved into
`cfg` first.

Three caveats belong next to the numbers. First, every run is a single seed. Second, the crop run differs from
`resized_512` in two ways at once: the resolution, and the rule that places half the crops on a filament. A run with
`pos_crop_prob=0.0` at 2048 would separate the two effects if the gap between the runs is large. Third, UNet++ and
DeepLabV3+ use the learning rate, loss and number of epochs chosen for the U-Net. That keeps the comparison controlled,
but it may not be the best setting for each architecture.